# VisionGuard cloud trial (Kaggle)

The recorded clip, an empty database, one administrator, a public https address.
Step by step: `specs/021-cloud-trial/quickstart.md`.

**Before Run all**: Settings → Accelerator **GPU T4**, Internet **on**. Inputs: your
private `visionguard-code` and `visionguard-models` datasets. Add-ons → Secrets:
`VG_ADMIN_PASSWORD`, attached to this notebook.

Nothing here contains a key. Ask and email report "not configured" in the trial.

In [ ]:
# 1. Copy the code from the code dataset to /tmp/vg (inputs are read-only).
import glob, os, shutil, subprocess, zipfile

VG = "/tmp/vg"
if not os.path.isdir(VG):
    scripts = glob.glob("/kaggle/input/**/tools/cloud/bootstrap.sh", recursive=True)
    zips = glob.glob("/kaggle/input/**/visionguard-trial.zip", recursive=True)
    if scripts:
        shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(scripts[0]))), VG)
    elif zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/tmp")
        os.rename("/tmp/visionguard", VG)
    else:
        raise SystemExit("No code found: add your visionguard-code dataset as an input.")
print("code in", VG, "-", len(os.listdir(VG)), "entries")

In [ ]:
# 2. The admin password, from the Kaggle secret. Never printed.
from kaggle_secrets import UserSecretsClient

env = dict(os.environ)
env["VG_ADMIN_PASSWORD"] = UserSecretsClient().get_secret("VG_ADMIN_PASSWORD")
env.pop("VG_DEV_SIGNIN", None)
print("secret read:", bool(env["VG_ADMIN_PASSWORD"]))

In [ ]:
# 3. Start: checks, installs, TensorRT, admin, tunnel, engine. Each step prints its time.
import time

log_path = "/tmp/vg-bootstrap.log"
log = open(log_path, "w")
boot = subprocess.Popen(["bash", f"{VG}/tools/cloud/bootstrap.sh"], cwd=VG, env=env,
                        stdout=log, stderr=subprocess.STDOUT)
shown = 0
while True:
    with open(log_path) as f:
        text = f.read()
    print(text[shown:], end="")
    shown = len(text)
    if boot.poll() is not None:
        break
    time.sleep(3)
print("\nbootstrap exit code:", boot.returncode)

In [ ]:
# 4. Keep this cell running while you use the console. It prints the engine's state
#    every 5 minutes, and stops when the engine stops. Interrupt it to get the
#    notebook back; the engine keeps running until you run cell 5 or time runs out.
import json, urllib.request

while True:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/healthz", timeout=10) as r:
            h = json.load(r)
        print(time.strftime("%H:%M:%S"), "engine up, stops at", h.get("endsAt"), flush=True)
    except Exception as exc:
        print(time.strftime("%H:%M:%S"), "engine not answering:", exc, flush=True)
        break
    time.sleep(300)

In [ ]:
# 5. Stop and save: database, logs, benchmark and summary go to /kaggle/working.
#    Download the trial-<date> folder from the Output panel before you close.
print(subprocess.run(["bash", f"{VG}/tools/cloud/stop.sh"], capture_output=True, text=True).stdout)